In [1]:
import numpy as np
import pandas as pd

In [2]:
from scipy.io import loadmat
true_mat = loadmat("corr-offsite-results_1-12_winter_hour_train60_multi1_th50.mat")
long_mat = loadmat("NWP.mat")
print(long_mat.keys())   
short_mat = loadmat("PER624.mat")
print(short_mat.keys())

dict_keys(['__header__', '__version__', '__globals__', 'NWP', 'sigma_nwp'])
dict_keys(['__header__', '__version__', '__globals__', 'maes_5', 'maes_6', 'pred', 'sigma_h'])


In [3]:
import numpy as np
from scipy.io import loadmat

true_mat  = loadmat("corr-offsite-results_1-12_winter_hour_train60_multi1_th50.mat")
long_mat  = loadmat("NWP.mat")
short_mat = loadmat("PER624.mat")

mu_long  = np.asarray(long_mat["NWP"])          
mu_long  = np.transpose(mu_long,  (2,1,0))      

mu_short = np.asarray(short_mat["pred"])
mu_short = np.transpose(mu_short, (2,1,0))     

y_raw  = np.asarray(true_mat["allY_Test"])     
y_true = np.transpose(y_raw, (2,1,0))           

muX = mu_long.astype(float)
muY = mu_short.astype(float)
y_true = y_true.astype(float)

D,L,H = muX.shape
assert muY.shape == (D,L,H)
assert y_true.shape == (D,L,H)
assert (L,H) == (2,24)
print("Shapes OK:", muX.shape, muY.shape, y_true.shape)

Shapes OK: (183, 2, 24) (183, 2, 24) (183, 2, 24)


In [4]:
def to_D_2_24(arr):
    arr = np.asarray(arr)
    if arr.shape == (24,2,D):
        return np.transpose(arr, (2,1,0)).astype(float)
    if arr.shape == (D,2,24):
        return arr.astype(float)
    raise ValueError(f"Unexpected shape {arr.shape}")

sigma_long  = to_D_2_24(long_mat["sigma_nwp"])
sigma_short = to_D_2_24(short_mat["sigma_h"])

SigXX = np.zeros((D,2,24,24), dtype=float)
SigYY = np.zeros((D,2,24,24), dtype=float)

for d in range(D):
    for loc in range(2):
        SigXX[d,loc] = np.diag(sigma_long[d,loc]**2)
        SigYY[d,loc] = np.diag(sigma_short[d,loc]**2)

In [5]:
import numpy as np
from scipy.optimize import minimize

I24 = np.eye(24)
Wwin = 45
S = 1000
M_pairs = 10000
seed = 42
rng_global = np.random.default_rng(seed)

eff = D - Wwin
muZ_out      = np.empty((24,2,eff))
SigZ_per_loc = np.empty((2,24,24,eff))
error_out    = np.empty((24,2,eff))
alpha_seq    = np.empty(eff)

def es_mc(mu_flat, Lbd, y_flat, eps_bank, i_pair, j_pair):
    X = mu_flat[None,:] + eps_bank @ Lbd.T
    term1 = np.mean(np.linalg.norm(X - y_flat[None,:], axis=1))
    diffs = X[i_pair] - X[j_pair]
    term2 = 0.5 * np.mean(np.linalg.norm(diffs, axis=1))
    return term1 - term2

alpha_lo, alpha_hi = 0, 1  
alpha_prev = 0.8

for d in range(Wwin, D):
    k   = d - Wwin
    win = np.arange(d - Wwin, d)
    Wlen= len(win)

    eps_bank = rng_global.standard_normal((Wlen, S, 48))
    i_pair   = rng_global.integers(0, S, size=M_pairs)
    j_pair   = rng_global.integers(0, S, size=M_pairs)
    m_pairs  = (i_pair != j_pair)
    i_pair, j_pair = i_pair[m_pairs], j_pair[m_pairs]

    def es_objective_alpha(alpha_arr):
        alpha = float(alpha_arr[0])
        w = 1 - np.exp(-alpha * (np.arange(24) + 1.0))
        Wmat = np.diag(w)
        IminusW = np.diag(1.0 - w)

        total = 0.0
        for kk, dd in enumerate(win):
            # loc0
            muZ0 = Wmat @ muX[dd,0] + IminusW @ muY[dd,0]
            SigZ0 = (Wmat @ SigXX[dd,0] @ Wmat
                     + IminusW @ SigYY[dd,0] @ IminusW)

            # loc1
            muZ1 = Wmat @ muX[dd,1] + IminusW @ muY[dd,1]
            SigZ1 = (Wmat @ SigXX[dd,1] @ Wmat
                     + IminusW @ SigYY[dd,1] @ IminusW)

            # chol (jitter)
            jitter = 1e-8
            for _ in range(8):
                try:
                    L0 = np.linalg.cholesky(SigZ0 + jitter*I24); break
                except np.linalg.LinAlgError:
                    jitter *= 10.0
            jitter = 1e-8
            for _ in range(8):
                try:
                    L1 = np.linalg.cholesky(SigZ1 + jitter*I24); break
                except np.linalg.LinAlgError:
                    jitter *= 10.0

            Lbd = np.zeros((48,48))
            Lbd[:24,:24] = L0
            Lbd[24:,24:] = L1

            mu_flat = np.concatenate([muZ0, muZ1], axis=0)
            y_flat  = np.concatenate([y_true[dd,0], y_true[dd,1]], axis=0)

            total += es_mc(mu_flat, Lbd, y_flat, eps_bank[kk], i_pair, j_pair)

        return total / Wlen

    opt = minimize(
        es_objective_alpha,
        x0=np.array([alpha_prev]),
        method="L-BFGS-B",
        bounds=[(alpha_lo, alpha_hi)],
        options=dict(maxiter=100, ftol=1e-9)
    )
    alpha_hat = float(opt.x[0])
    alpha_prev = alpha_hat
    alpha_seq[k] = alpha_hat

    w = 1.0 - np.exp(-alpha_hat * (np.arange(24) + 1.0))
    IminusW = np.diag(1.0 - w)

    Wmat = np.diag(w)

    for loc in range(2):
        muZ = Wmat @ muX[d,loc] + IminusW @ muY[d,loc] 
        SigZ = (Wmat @ SigXX[d,loc] @ Wmat
                + IminusW @ SigYY[d,loc] @ IminusW)

        muZ_out[:,loc,k] = muZ
        SigZ_per_loc[loc,:,:,k] = SigZ
        error_out[:,loc,k] = y_true[d,loc] - muZ

    if (k+1) % 10 == 0 or k == eff-1:
        mae = float(np.abs(error_out[:,:,k]).mean())
        print(f"[{k+1}/{eff}] alpha={alpha_hat:.4f}  MAE={mae:.4f}")

[10/138] alpha=0.3537  MAE=1.6331
[20/138] alpha=0.4571  MAE=1.0875
[30/138] alpha=0.3980  MAE=3.7524
[40/138] alpha=0.5334  MAE=1.7232
[50/138] alpha=0.5341  MAE=1.1900
[60/138] alpha=0.4961  MAE=1.4003
[70/138] alpha=0.4579  MAE=1.0192
[80/138] alpha=0.4139  MAE=1.5880
[90/138] alpha=0.3363  MAE=1.6375
[100/138] alpha=0.3127  MAE=1.2965
[110/138] alpha=0.2858  MAE=1.0398
[120/138] alpha=0.2995  MAE=1.7908
[130/138] alpha=0.2672  MAE=1.8790
[138/138] alpha=0.1978  MAE=0.7457


In [ ]:
from scipy.io import savemat

## save the file
savemat("ensemble_nwp_per.mat", {
    "mu": muZ_out,
    "Sigma": SigZ_per_loc,
    "Error": error_out,
    "alpha": alpha_seq,
    "y_true": y_true   
})
